In [1]:
import uuid
import bs4

from typing import Iterator
from langchain_core.documents import Document
from langchain_community.document_loaders import AsyncHtmlLoader

from question_answer import QuestionAnswer

def extract_faq(url: str) -> Iterator[Document]:
    for doc in AsyncHtmlLoader([url]).load():
        soup = bs4.BeautifulSoup(doc.page_content, "html.parser")
        for q in soup.find_all(class_="card-faq"):
            text     = q.get_text().strip()
            splits   = text.split("\n\n\n\n")
            question = splits[0].strip()
            answer   = splits[1].strip()
            
            if answer.endswith("\n\nCopy link"):
                answer = answer[:-len("\n\nCopy link")].strip()

            mdtxt = f"## {question}\n\n{answer}"
            yield Document(page_content=question, metadata = {"fulltext": mdtxt, **doc.metadata})



USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
url = "https://agatha.arch.be/help/"
faq = list(extract_faq(url))

Fetching pages: 100%|###########################################################| 1/1 [00:00<00:00, 11.06it/s]


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

db = Chroma(
    "faq", 
    HuggingFaceEmbeddings(
        model_name   = "intfloat/multilingual-e5-large-instruct",
        model_kwargs = {"device": "cuda", "model_kwargs": {"torch_dtype": "float16"}, },
    ),
    persist_directory="./digita",
)
db.add_documents(faq)

['9ce69ff6-0ad7-4ad6-a734-678ee5bec263',
 'ffc519f2-0ce1-4d80-b7d1-ac5c24353ab4',
 'e13e59e2-0ba8-4350-b92f-ed77a14e7742',
 '89f76296-86e0-403d-a5bf-b0888cedc26e',
 '91fd44cd-cccc-4319-898b-cfbc31929131',
 '6ea50c81-b4ca-4cf5-ba1f-a318b0bb0965',
 'f6bbd7b7-3181-47c9-8de0-005d11040be2',
 '19102442-be03-4a6c-9af2-740f5eb335b3',
 '50d7a250-12a6-4453-a55d-0ec1a7f7b7a3',
 '68d48e19-edaf-4577-8276-737bd1781e06',
 '6f94da8b-7366-44e8-8398-105a5b3fa68c',
 'd9fa46b5-c44d-4e45-a9ae-cc43a4fbb1ae',
 '67777e75-2e0b-4798-83e4-49d974c90f2a',
 'b9d1292a-4f1a-4529-8281-5a8aeb26b0af',
 'ef918967-5330-4a40-bc94-b6c65a8af5f1',
 '90ac68fe-0a7c-4664-892e-a807d4f77ccf',
 '8adff53d-5743-4768-b343-57e7f1b09c90',
 'de6f5e45-ddd4-4385-ad83-eb4d0332e65d',
 '7ac7f03a-6613-449e-9762-ea6551c506f8',
 '25029c34-6da9-488c-b0f4-7546fb7ccad6',
 'c76b1f01-9edc-4188-9faf-cd709711168f',
 'dc6ec4a4-07c1-43f4-95d5-fbbf2f644c6f',
 '0fbfb444-bbf5-49bb-8568-eaf7255a65f6',
 'cb37a0f0-5af3-4362-a3ff-71518b638fd6',
 '60ae2ef0-8dcc-